In [10]:
import pandas as pd
import numpy as np

In [13]:
from google.colab import files
uploaded = files.upload()

Saving KDDTrain+ (1).txt to KDDTrain+ (1).txt


In [14]:
from google.colab import files
uploaded = files.upload()

Saving KDDTest+ (1).txt to KDDTest+ (1).txt


In [15]:
columns = [
"duration","protocol_type","service","flag","src_bytes","dst_bytes",
"land","wrong_fragment","urgent","hot","num_failed_logins",
"logged_in","num_compromised","root_shell","su_attempted",
"num_root","num_file_creations","num_shells","num_access_files",
"num_outbound_cmds","is_host_login","is_guest_login","count",
"srv_count","serror_rate","srv_serror_rate","rerror_rate",
"srv_rerror_rate","same_srv_rate","diff_srv_rate","srv_diff_host_rate",
"dst_host_count","dst_host_srv_count","dst_host_same_srv_rate",
"dst_host_diff_srv_rate","dst_host_same_src_port_rate",
"dst_host_srv_diff_host_rate","dst_host_serror_rate",
"dst_host_srv_serror_rate","dst_host_rerror_rate",
"dst_host_srv_rerror_rate","label","difficulty"
]

In [31]:
train = pd.read_csv("KDDTrain+ (1).txt", names = columns)
test = pd.read_csv("KDDTest+ (1).txt", names = columns)

In [32]:
train.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [33]:
train.shape

(125973, 43)

In [23]:
train.head()


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [24]:
train.shape

(125973, 43)

In [17]:
train['label'].value_counts()

,count
label,
normal,67343
neptune,41214
satan,3633
ipsweep,3599
portsweep,2931
smurf,2646
nmap,1493
back,956
teardrop,892


In [18]:
train['label'].value_counts().head()

,count
label,
normal,67343
neptune,41214
satan,3633
ipsweep,3599
portsweep,2931


In [34]:
train['label'] = train['label'].apply(lambda x: 0 if str(x).strip() == 'normal' else 1)
test['label'] = test['label'].apply(lambda x: 0 if str(x).strip() == 'normal' else 1)

In [28]:
train['label'].value_counts()

,count
label,
0,67343
1,58630


In [35]:
train.drop('difficulty', axis=1, inplace=True)
test.drop('difficulty', axis=1, inplace=True)

In [31]:
train.shape

(125973, 42)

In [36]:
train = pd.get_dummies(train, columns=['protocol_type','service','flag'])
test = pd.get_dummies(test, columns=['protocol_type','service','flag'])

In [33]:
train.shape

(125973, 123)

In [37]:
X_train = train.drop('label', axis=1)
y_train = train['label']

X_test = test.drop('label', axis=1)
y_test = test['label']

In [27]:
X_train.shape

(125973, 41)

In [38]:
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [39]:
from sklearn.preprocessing import MinMaxScaler

In [40]:
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)

In [41]:
X_test = scaler.transform(X_test)


In [40]:
X_train.shape

(125973, 122)

In [42]:
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

X_train.shape


(125973, 122, 1)

In [43]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout

In [44]:
model = Sequential()

model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(MaxPooling1D(pool_size=2))
model.add(LSTM(64))
model.add(Dropout(0.5))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [45]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [46]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 120, 64)        │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 35,393 (138.25 KB)

 Trainable params: 35,393 (138.25 KB)

 Non-trainable params: 0 (0.00 B)

In [47]:
y_train = y_train.astype('float32')
y_test = y_test.astype('float32')
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

In [16]:
print(type(X_train))
print(type(y_train))
print(type(X_test))
print(type(y_test))

print(X_train.shape)
print(y_train.shape)
print(X_train.dtype)
print(y_train.dtype)

<class 'numpy.ndarray'>
<class 'pandas.core.series.Series'>
<class 'numpy.ndarray'>
<class 'pandas.core.series.Series'>
(125973, 122, 1)
(125973,)
float32
float32


In [48]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/5
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 89s 28ms/step - accuracy: 0.9455 - loss: 0.1623 - val_accuracy: 0.9701 - val_loss: 0.0887
Epoch 2/5
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 139s 27ms/step - accuracy: 0.9745 - loss: 0.0842 - val_accuracy: 0.9759 - val_loss: 0.0690
Epoch 3/5
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 142s 27ms/step - accuracy: 0.9765 - loss: 0.0666 - val_accuracy: 0.9799 - val_loss: 0.0531
Epoch 4/5
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 143s 27ms/step - accuracy: 0.9785 - loss: 0.0576 - val_accuracy: 0.9813 - val_loss: 0.0555
Epoch 5/5
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 82s 26ms/step - accuracy: 0.9825 - loss: 0.0492 - val_accuracy: 0.9792 - val_loss: 0.0572


In [49]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_accuracy)

705/705 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.7296 - loss: 1.1381
Test Accuracy: 0.7295510768890381


In [50]:
y_pred = (model.predict(X_test) > 0.5).astype("int32")

705/705 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step


In [51]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, classification_report

In [52]:
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Precision: 0.9758406329471602
Recall: 0.5382217719940777
F1 Score: 0.6937873537240721


In [53]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[8882  829]
 [3732 9101]]


In [54]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.70      0.91      0.80      9711
         1.0       0.92      0.71      0.80     12833

    accuracy                           0.80     22544
   macro avg       0.81      0.81      0.80     22544
weighted avg       0.83      0.80      0.80     22544



In [53]:
import tensorflow as tf

In [54]:
loss_object = tf.keras.losses.BinaryCrossentropy()

def fgsm_attack(model, x, y, epsilon):
    x_tensor = tf.convert_to_tensor(x, dtype=tf.float32)
    y_tensor = tf.convert_to_tensor(y, dtype=tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(x_tensor)
        prediction = model(x_tensor)
        loss = loss_object(y_tensor, prediction)

    gradient = tape.gradient(loss, x_tensor)
    signed_grad = tf.sign(gradient)

    x_adv = x_tensor + epsilon * signed_grad
    x_adv = tf.clip_by_value(x_adv, 0, 1)

    return x_adv.numpy()

In [55]:
X_test_small = X_test[:500]
y_test_small = y_test[:500]
epsilon = 0.01
X_test_fgsm = fgsm_attack(model, X_test_small, y_test_small, epsilon)

In [56]:
fgsm_loss, fgsm_accuracy = model.evaluate(X_test_fgsm, y_test_small)
print("FGSM Test Accuracy:", fgsm_accuracy)

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7560 - loss: 1.0519
FGSM Test Accuracy: 0.7559999823570251


In [57]:
y_pred_fgsm = (model.predict(X_test_fgsm) > 0.5).astype("int32").ravel()

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step


In [58]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, classification_report

print("FGSM Precision:", precision_score(y_test_small, y_pred_fgsm))
print("FGSM Recall:", recall_score(y_test_small, y_pred_fgsm))
print("FGSM F1-score:", f1_score(y_test_small, y_pred_fgsm))

cm_fgsm = confusion_matrix(y_test_small, y_pred_fgsm)
print(cm_fgsm)

FGSM Precision: 0.935672514619883
FGSM Recall: 0.5904059040590406
FGSM F1-score: 0.7239819004524887
[[218  11]
 [111 160]]


In [59]:
attack_indices = (y_test_small == 1)
predicted_attacks = y_pred_fgsm[attack_indices]

esr = ((predicted_attacks == 0).sum() / len(predicted_attacks)) * 100
print("FGSM ESR:", esr)

FGSM ESR: 40.959409594095945


In [60]:
batch_size = 128
epsilon = 0.01

fgsm_batches = []

for i in range(0, len(X_test), batch_size):
    x_batch = X_test[i:i+batch_size]
    y_batch = y_test[i:i+batch_size]

    x_adv_batch = fgsm_attack(model, x_batch, y_batch, epsilon)
    fgsm_batches.append(x_adv_batch)

    if i % 2000 == 0:
        print(f"Processed {i} samples")

X_test_fgsm_full = np.concatenate(fgsm_batches, axis=0).astype("float32")

Processed 0 samples
Processed 16000 samples


In [61]:
fgsm_loss_full, fgsm_accuracy_full = model.evaluate(X_test_fgsm_full, y_test, verbose=0)
print("Full FGSM Test Accuracy:", fgsm_accuracy_full)

Full FGSM Test Accuracy: 0.7131831049919128


In [62]:
y_pred_fgsm_full = (model.predict(X_test_fgsm_full, batch_size=128, verbose=0) > 0.5).astype("int32").ravel()

In [63]:
fgsm_precision_full = precision_score(y_test, y_pred_fgsm_full)
fgsm_recall_full = recall_score(y_test, y_pred_fgsm_full)
fgsm_f1_full = f1_score(y_test, y_pred_fgsm_full)
cm_fgsm_full = confusion_matrix(y_test, y_pred_fgsm_full)

print("FGSM Precision:", fgsm_precision_full)
print("FGSM Recall:", fgsm_recall_full)
print("FGSM F1-score:", fgsm_f1_full)
print(cm_fgsm_full)

FGSM Precision: 0.9527805433082065
FGSM Recall: 0.5220135587937349
FGSM F1-score: 0.6744865082561418
[[9379  332]
 [6134 6699]]


In [64]:
attack_mask = (y_test == 1)
esr_full = ((y_pred_fgsm_full[attack_mask] == 0).sum() / attack_mask.sum()) * 100
print("Full FGSM ESR:", esr_full)

Full FGSM ESR: 47.79864412062651


In [36]:
fgsm_results = pd.DataFrame([{
    "Condition": "Baseline CNN-LSTM on FGSM test",
    "Epsilon": epsilon,
    "Accuracy": fgsm_accuracy_full,
    "Precision": fgsm_precision_full,
    "Recall": fgsm_recall_full,
    "F1-score": fgsm_f1_full,
    "ESR": esr_full
}])

fgsm_results

,Condition,Epsilon,Accuracy,Precision,Recall,F1-score,ESR
0,Baseline CNN-LSTM on FGSM test,0.01,0.768009,0.918898,0.649809,0.761274,35.019091


In [37]:
np.save("X_test_fgsm_full.npy", X_test_fgsm_full)
np.save("y_pred_fgsm_full.npy", y_pred_fgsm_full)

In [65]:
def pgd_attack(model, x, y, epsilon=0.01, alpha=0.002, iterations=10):
    x_orig = tf.convert_to_tensor(x, dtype=tf.float32)
    x_adv = tf.identity(x_orig)
    y_tensor = tf.convert_to_tensor(y, dtype=tf.float32)

    for _ in range(iterations):
        with tf.GradientTape() as tape:
            tape.watch(x_adv)
            prediction = model(x_adv)
            loss = loss_object(y_tensor, prediction)

        gradient = tape.gradient(loss, x_adv)
        signed_grad = tf.sign(gradient)

        # small iterative step
        x_adv = x_adv + alpha * signed_grad

        # project back into epsilon-ball around original input
        x_adv = tf.clip_by_value(x_adv, x_orig - epsilon, x_orig + epsilon)

        # keep valid normalized range
        x_adv = tf.clip_by_value(x_adv, 0, 1)

    return x_adv.numpy()


In [66]:
X_test_small = X_test[:500]
y_test_small = y_test[:500]

X_test_pgd = pgd_attack(model, X_test_small, y_test_small, epsilon=0.01, alpha=0.002, iterations=10)
print("PGD subset shape:", X_test_pgd.shape)

PGD subset shape: (500, 122, 1)


In [68]:
pgd_loss, pgd_accuracy = model.evaluate(X_test_pgd, y_test_small, verbose=0)
print("PGD Test Accuracy:", pgd_accuracy)

PGD Test Accuracy: 0.7559999823570251


In [69]:
y_pred_pgd = (model.predict(X_test_pgd, verbose=0) > 0.5).astype("int32").ravel()

print("PGD Precision:", precision_score(y_test_small, y_pred_pgd))
print("PGD Recall:", recall_score(y_test_small, y_pred_pgd))
print("PGD F1-score:", f1_score(y_test_small, y_pred_pgd))

cm_pgd = confusion_matrix(y_test_small, y_pred_pgd)
print(cm_pgd)

PGD Precision: 0.935672514619883
PGD Recall: 0.5904059040590406
PGD F1-score: 0.7239819004524887
[[218  11]
 [111 160]]


In [70]:
attack_indices = (y_test_small == 1)
predicted_attacks = y_pred_pgd[attack_indices]

pgd_esr = ((predicted_attacks == 0).sum() / len(predicted_attacks)) * 100
print("PGD ESR:", pgd_esr)

PGD ESR: 40.959409594095945


In [71]:
batch_size = 64
pgd_batches = []

for i in range(0, len(X_test), batch_size):
    x_batch = X_test[i:i+batch_size]
    y_batch = y_test[i:i+batch_size]

    x_adv_batch = pgd_attack(model, x_batch, y_batch, epsilon=0.01, alpha=0.002, iterations=10)
    pgd_batches.append(x_adv_batch)

    if i % 2000 == 0:
        print(f"Processed {i} samples")

X_test_pgd_full = np.concatenate(pgd_batches, axis=0).astype("float32")
print("Full PGD shape:", X_test_pgd_full.shape)

Processed 0 samples
Processed 8000 samples
Processed 16000 samples
Full PGD shape: (22544, 122, 1)


In [72]:
pgd_loss_full, pgd_accuracy_full = model.evaluate(X_test_pgd_full, y_test, verbose=0)
print("Full PGD Accuracy:", pgd_accuracy_full)

Full PGD Accuracy: 0.7072391510009766


In [74]:
y_pred_pgd_full = (model.predict(X_test_pgd_full, batch_size=128, verbose=0) > 0.5).astype("int32").ravel()

In [75]:
pgd_precision_full = precision_score(y_test, y_pred_pgd_full)
pgd_recall_full = recall_score(y_test, y_pred_pgd_full)
pgd_f1_full = f1_score(y_test, y_pred_pgd_full)
cm_pgd_full = confusion_matrix(y_test, y_pred_pgd_full)

print("PGD Precision:", pgd_precision_full)
print("PGD Recall:", pgd_recall_full)
print("PGD F1-score:", pgd_f1_full)
print(cm_pgd_full)

PGD Precision: 0.9367904695164682
PGD Recall: 0.520844697264864
PGD F1-score: 0.6694711538461539
[[9260  451]
 [6149 6684]]


In [76]:
attack_mask = (y_test == 1)
pgd_esr_full = ((y_pred_pgd_full[attack_mask] == 0).sum() / attack_mask.sum()) * 100
print("Full PGD ESR:", pgd_esr_full)


Full PGD ESR: 47.915530273513596
